# Superstore — Exploratory Analysis

**Phase 1:** Sales & Profitability  
**Dataset:** Superstore (Kaggle: `vivek468/superstore-dataset-final`)  
**Database:** `../data/superstore.db`  
**SQL scripts:** `../sql/`

---

### Contents
1. Global KPIs
2. Performance by Category
3. Discount Impact on Profit Margin
4. Regional Performance
5. Annual Trends & Monthly Seasonality
6. Customer Segments & Ship Mode
7. Product Analysis
8. Customer Analysis

In [ ]:
import sqlite3
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option('display.float_format', '{:,.1f}'.format)
pd.set_option('display.max_colwidth', 60)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

In [ ]:
DB_PATH  = '../data/superstore.db'
SQL_PATH = '../sql/'

conn = sqlite3.connect(DB_PATH)

def run_sql_file(filepath):
    """Read a .sql file, strip comments, and return a list of DataFrames — one per statement."""
    with open(filepath) as f:
        content = f.read()
    content = re.sub(r'--[^\n]*', '', content)          # strip line comments
    statements = [s.strip() for s in content.split(';') if s.strip()]
    return [pd.read_sql_query(s, conn) for s in statements]

def fmt_usd(val):
    return f'${val:,.0f}'

print('Connected to', DB_PATH)

---
## 1. Global KPIs

In [ ]:
results = run_sql_file(SQL_PATH + '01_global_metrics.sql')
kpis = results[0].T
kpis.columns = ['Value']
kpis

---
## 2. Performance by Category

In [ ]:
results = run_sql_file(SQL_PATH + '02_by_category.sql')
cat_df  = results[0]
sub_df  = results[1]

print('=== Category ===')
display(cat_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].barh(cat_df['Category'], cat_df['sales'], color='steelblue')
axes[0].set_title('Sales by Category')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

axes[1].barh(cat_df['Category'], cat_df['margin_pct'],
             color=['#d62728' if v < 5 else 'steelblue' for v in cat_df['margin_pct']])
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Profit Margin % by Category')
axes[1].set_xlabel('%')

plt.tight_layout()
plt.savefig('../visuals/02a_category_performance.png', bbox_inches='tight')
plt.show()

In [ ]:
print('=== Sub-Category (sorted by profit) ===')
display(sub_df)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#d62728' if v < 0 else 'steelblue' for v in sub_df['profit']]
ax.barh(sub_df['Sub-Category'], sub_df['profit'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Profit by Sub-Category')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
plt.tight_layout()
plt.savefig('../visuals/02b_subcategory_profit.png', bbox_inches='tight')
plt.show()

---
## 3. Discount Impact on Profit Margin

> **Breakeven point ~20% discount.** Every discount ≥ 20% produces negative average margin.

In [ ]:
results  = run_sql_file(SQL_PATH + '03_discount_impact.sql')
disc_df  = results[0]
display(disc_df)

fig, ax1 = plt.subplots(figsize=(10, 5))

bar_colors = ['#d62728' if v < 0 else 'steelblue' for v in disc_df['margin_pct']]
bars = ax1.bar(disc_df['discount_bucket'], disc_df['items'], color='lightsteelblue', label='Items')
ax1.set_ylabel('Items')
ax1.set_xlabel('Discount Bucket')

ax2 = ax1.twinx()
ax2.plot(disc_df['discount_bucket'], disc_df['margin_pct'], color='crimson',
         marker='o', linewidth=2, label='Margin %')
ax2.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax2.set_ylabel('Profit Margin %')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.title('Discount Bucket: Item Count vs Profit Margin %')
plt.tight_layout()
plt.savefig('../visuals/03_discount_impact.png', bbox_inches='tight')
plt.show()

---
## 4. Regional Performance

In [ ]:
results    = run_sql_file(SQL_PATH + '04_by_region.sql')
region_df  = results[0]
states_df  = results[1]

print('=== Region ===')
display(region_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].barh(region_df['Region'], region_df['sales'], color='steelblue')
axes[0].set_title('Sales by Region')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

axes[1].barh(region_df['Region'], region_df['margin_pct'], color='steelblue')
axes[1].set_title('Profit Margin % by Region')
axes[1].set_xlabel('%')

plt.tight_layout()
plt.savefig('../visuals/04a_region_performance.png', bbox_inches='tight')
plt.show()

In [ ]:
print('=== States with Highest Losses ===')
display(states_df)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d62728' if v < 0 else 'steelblue' for v in states_df['profit']]
ax.barh(states_df['State'], states_df['profit'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Profit by State (Bottom 10)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
plt.tight_layout()
plt.savefig('../visuals/04b_state_losses.png', bbox_inches='tight')
plt.show()

---
## 5. Annual Trends & Monthly Seasonality

In [ ]:
results    = run_sql_file(SQL_PATH + '05_trends.sql')
annual_df  = results[0]
monthly_df = results[1]

print('=== Annual Trend ===')
display(annual_df)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.bar(annual_df['year'], annual_df['sales'], color='lightsteelblue', label='Sales')
ax1.set_ylabel('Sales ($)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

ax2 = ax1.twinx()
ax2.plot(annual_df['year'], annual_df['margin_pct'], color='crimson',
         marker='o', linewidth=2, label='Margin %')
ax2.set_ylabel('Margin %')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('Annual Sales & Profit Margin (2014–2017)')
plt.tight_layout()
plt.savefig('../visuals/05a_annual_trend.png', bbox_inches='tight')
plt.show()

In [ ]:
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly_df['month_name'] = monthly_df['month'].apply(lambda m: month_labels[m - 1])

print('=== Monthly Seasonality ===')
display(monthly_df[['month_name','items','sales','profit','margin_pct']])

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(monthly_df['month_name'], monthly_df['sales'], color='steelblue')
ax.set_title('Monthly Sales (all years combined)')
ax.set_ylabel('Sales ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
plt.tight_layout()
plt.savefig('../visuals/05b_monthly_seasonality.png', bbox_inches='tight')
plt.show()

---
## 6. Customer Segments & Ship Mode

In [ ]:
results   = run_sql_file(SQL_PATH + '06_by_segment.sql')
seg_df    = results[0]
ship_df   = results[1]

print('=== Segment ===')
display(seg_df)
print('\n=== Ship Mode ===')
display(ship_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].barh(seg_df['Segment'], seg_df['margin_pct'], color='steelblue')
axes[0].set_title('Profit Margin % by Segment')
axes[0].set_xlabel('%')

axes[1].barh(ship_df['Ship Mode'], ship_df['margin_pct'], color='steelblue')
axes[1].set_title('Profit Margin % by Ship Mode')
axes[1].set_xlabel('%')

plt.tight_layout()
plt.savefig('../visuals/06_segment_shipmode.png', bbox_inches='tight')
plt.show()

---
## 7. Product Analysis

In [ ]:
results      = run_sql_file(SQL_PATH + '07_products.sql')
top_prod_df  = results[0]
loss_prod_df = results[1]
count_loss   = results[2]

print('=== Top 10 Products by Profit ===')
display(top_prod_df)

print('\n=== Top 10 Products with Highest Losses ===')
display(loss_prod_df)

print(f"\nProducts with negative cumulative profit: {count_loss['products_at_loss'].iloc[0]}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(top_prod_df['Product Name'].str[:40], top_prod_df['profit'], color='steelblue')
axes[0].set_title('Top 10 Products by Profit')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

axes[1].barh(loss_prod_df['Product Name'].str[:40], loss_prod_df['profit'], color='#d62728')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Top 10 Products — Highest Losses')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

plt.tight_layout()
plt.savefig('../visuals/07_product_analysis.png', bbox_inches='tight')
plt.show()

---
## 8. Customer Analysis

In [ ]:
results    = run_sql_file(SQL_PATH + '08_customers.sql')
top_cust   = results[0]
cust_dist  = results[1]

print('=== Top 20 Customers by Sales ===')
display(top_cust)

print('\n=== Customer Profit Distribution ===')
display(cust_dist)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d62728' if v < 0 else 'steelblue' for v in top_cust['profit']]
ax.barh(top_cust['Customer Name'], top_cust['profit'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 20 Customers by Sales — Profit ($)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
plt.tight_layout()
plt.savefig('../visuals/08_customer_profit.png', bbox_inches='tight')
plt.show()

---
## Key Findings — Phase 1 Summary

| Finding | Detail |
|---------|--------|
| Global margin | 12.47% across $2.3M in sales |
| Furniture problem | 32% of sales, only 6.4% of profit (2.5% margin) |
| Discount breakeven | ≥20% discount → negative average margin |
| Worst region | Central: 7.9% margin vs West 14.9% |
| Worst state | Texas: -$25,729 profit |
| Sales growth | +51% over 4 years, but margin dipped in 2017 |
| Peak season | Q4 (Sep–Dec): highest sales and profit |
| Value destroyers | Tables (-$17.7K), Bookcases (-$3.5K), Cubify printers |

**Next:** Phase 2 — business interpretation and actionable recommendations.

In [ ]:
conn.close()
print('Done. Visuals saved to ../visuals/')